## Production_Coding -> Logging -> type hint -> dotenv 

##### **I am going to practice using logging instead of print statements because that's what we use in production. I'll also practice type hints and python-dotenv.**

#### Logging:

In [2]:
# logging
import logging
logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(levelname)s - %(messages)s') # %(name)s, %(filename)s,%(lineno)s
logger = logging.getLogger(__name__)
logger.debug("It will run in dev time")
logger.info("App started successfully")
logger.warning("Config file is missing,using default")
logger.error("API call failed",exc_info=True) # exc_info show the error with full history of the app function
logger.critical("Database down")

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\aswat\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\logging\__init__.py", line 449, in format
    return self._format(record)
           ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\aswat\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\logging\__init__.py", line 445, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
KeyError: 'messages'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\aswat\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\logging\__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\aswat\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\logging\__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\aswat\AppData\Roaming\uv\python\cpython-3.11

### LEVELS(severity Order):DEBUG < INFO < WARNING < ERROR < CRITICAL

**Handlers**

In [5]:
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

In [ ]:
# in real-time log file exceeds more than 10mb it will create new log file auromatically
from logging.handlers import RotatingFileHandler

handler = RotatingFileHandler("app.log", maxBytes=10_000_000, backupCount=5)

#### Type-Hints:

In [3]:
from typing import List, Dict, Optional, Union
def get_scores(names:List[str]) -> Dict[str,int]:
    return {name: len(name) for name in names}
def find_user(user_id:int) -> Optional[str]:
    #optional either it will return str or none 
    if user_id == 1:
        return "aswath"
    return None
def process(value: Union[str,int]) -> str:
    # Union returns int or str
    return str(value)

#### Dot-env:

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
print(api_key)

None


##### Type Annotations - Recap + Deep dive

In [6]:
#Annotation = broader concept of typing - It fix the "type label" for variable/function 
name: str = "Aswath"          # variable annotation
age: int = 25
scores: list[int] = [90, 85]

def greet(name: str, age: int = 25) -> str:   # function annotation
    return f"{name} is {age}"

#### dataclass - "Auto-generate boring code"

In [7]:
# if we use dataclass decorator we dont want to mention the __init__,__repr__,__eq__ functions because it automatically generate that

from dataclasses import dataclass
@dataclass
class User:
    name: str
    age: int
u = User("Aswath",25)
print(u)
print(u.name)

User(name='Aswath', age=25)
Aswath


In [ ]:
from dataclasses import dataclass,field
@dataclass
class Config:
    name: str
    log_level:str ="INFO"
    tags:list[str]=field(default_factory=list) # list and dict are mutable object so we cannot give more default values so we are using default_factory

#### Pydantic - "Dataclass + Validation"

##### why we are using pydantic means in data class it wont validate the input it only type-hint it but pydantic do the both things

In [3]:
from pydantic import BaseModel,Field # Field-> extra validation rule

class User(BaseModel):
    name:str
    age:int
    api_request:int = Field(default=1000,gt=0)
u=User(name="aswath",age=25,api_request=100)
print(u)

name='aswath' age=25 api_request=100


In [4]:
# Real_time EDA scenerio:
from pydantic import BaseModel,Field
from typing import Optional

class api_config(BaseModel):
    api_key:str
    model_name:str="claude-haiku"
    max_token:int = Field(default=10000,gt=0)
    temperature:Optional[float]=None

app = api_config(api_key="abcdef",max_token=500)
print(app.model_dump()) # convert object to dict format

{'api_key': 'abcdef', 'model_name': 'claude-haiku', 'max_token': 500, 'temperature': None}


** Internal, trusted data → dataclass. External/untrusted data (user input, API response, config files) → pydantic.**